In [1]:
import sys
import os
# 获取当前Notebook的绝对路径
notebook_dir = os.path.abspath('')
# 项目根目录是Notebook所在目录的父目录
project_root = os.path.abspath(os.path.join(notebook_dir, "../../.."))
# 将项目根目录添加到sys.path
if project_root not in sys.path:
    sys.path.insert(0, project_root)
# 确认路径是否正确
print("当前工作目录:", os.getcwd())
print("项目根目录:", project_root)
print("sys.path:", sys.path)
from apt.vendor.tspro.base import base as base
from apt.os.minio.MinioHandler import MinioClientWrapper

# 创建客户端
minio_client = MinioClientWrapper()

当前工作目录: c:\Users\GHUIQ\repos\MyFunds\jupyterbook\os\minio
项目根目录: c:\Users\GHUIQ\repos\MyFunds
sys.path: ['c:\\Users\\GHUIQ\\repos\\MyFunds', 'c:\\Users\\GHUIQ\\repos\\MyFunds\\.conda\\python311.zip', 'c:\\Users\\GHUIQ\\repos\\MyFunds\\.conda\\DLLs', 'c:\\Users\\GHUIQ\\repos\\MyFunds\\.conda\\Lib', 'c:\\Users\\GHUIQ\\repos\\MyFunds\\.conda', '', 'c:\\Users\\GHUIQ\\repos\\MyFunds\\.conda\\Lib\\site-packages', 'c:\\Users\\GHUIQ\\repos\\MyFunds\\.conda\\Lib\\site-packages\\win32', 'c:\\Users\\GHUIQ\\repos\\MyFunds\\.conda\\Lib\\site-packages\\win32\\lib', 'c:\\Users\\GHUIQ\\repos\\MyFunds\\.conda\\Lib\\site-packages\\Pythonwin']


# MinioHandler 使用演示

## 更新说明 (2025-07-13)

本演示已适配新的环境变量配置：

- **环境变量名称更新**: 从 `ENDPOINT`、`ACCESS_KEY`、`SECRET_KEY`、`CACHE_PATH` 更新为 `minio_ENDPOINT`、`minio_ACCESS_KEY`、`minio_SECRET_KEY`、`minio_CACHE_PATH`
- **配置文件位置**: 配置信息存储在项目根目录的 `.env` 文件中
- **当前配置**:
  - minio_ENDPOINT = "192.168.1.201:19000"
  - minio_ACCESS_KEY = "A9LUdGKJSbSTyV9FrfsF"
  - minio_SECRET_KEY = "dyyJTGH2bhqXp2ziyIzbJ8os3GOuEbPDauvdMq60"
  - minio_CACHE_PATH = "c:\minio_cache"

## 功能演示

本 Notebook 演示了 MinioClientWrapper 的主要功能：

1. **初始化和连接** - 使用环境变量自动配置
2. **桶管理** - 列出、创建、删除桶
3. **文件管理** - 上传、下载、列出文件
4. **配置验证** - 验证环境变量和连接状态

In [2]:
# 列出所有的桶及其信息
minio_client.list_buckets()

,桶名称,创建日期,总文件数,总大小
0,hdf5,2024-12-19 05:48,5254,80.7 GB
1,share,2024-12-19 05:49,535,1.9 GB


In [3]:
# 新建和删除桶
bucket_name = "test"
minio_client.create_bucket(bucket_name)
print("创建桶后:")
print(minio_client.list_buckets())

minio_client.delete_bucket(bucket_name)
print("删除桶后:")
print(minio_client.list_buckets())

创建桶后:
     桶名称              创建日期  总文件数      总大小
0   hdf5  2024-12-19 05:48  5254  80.7 GB
1  share  2024-12-19 05:49   535   1.9 GB
2   test  2025-07-13 13:30     0    0.0 B
删除桶后:
     桶名称              创建日期  总文件数      总大小
0   hdf5  2024-12-19 05:48  5254  80.7 GB
1  share  2024-12-19 05:49   535   1.9 GB


In [4]:
# 列出桶的文件列表
bucket_name = "hdf5"
# 列出某一个或多个文件 使用prefix做过滤
#minio_client.list_files(bucket_name,prefix="akshare/data/1min/000001.SZ.h5")
# 列出全部文件
minio_client.list_files(bucket_name)

,文件名称,修改日期,文件大小
0,akshare/data/1min/000001.SZ.h5,2024-12-24 12:29:25,31.6MB
1,akshare/data/1min/000002.SZ.h5,2024-12-24 12:29:30,31.1MB
2,akshare/data/1min/000004.SZ.h5,2024-12-24 12:29:24,22.0MB
3,akshare/data/1min/000005.SZ.h5,2024-12-24 12:29:23,22.4MB
4,akshare/data/1min/000006.SZ.h5,2024-12-24 12:29:27,28.9MB
...,...,...,...
5249,akshare/data/1min_test/000008.SZ.h5,2024-12-24 13:13:17,21.8MB
5250,akshare/data/1min_test/000009.SZ.h5,2024-12-24 13:13:19,31.2MB
5251,akshare/data/1min_test/000010.SZ.h5,2024-12-24 13:13:20,19.5MB
5252,akshare/data/1min_test/600000_test.SH.h5,2025-01-08 16:08:18,30.6MB


In [ ]:
# 上传文件
import tkinter as tk
from tkinter import filedialog
bucket_name = "share"
# 创建一个隐藏的Tkinter窗口
root = tk.Tk()
root.withdraw()
# 弹出文件选择对话框
file_path = filedialog.askopenfilename()
file_name = os.path.basename(file_path)
# 上传文件到Minio
if file_path:
    minio_client.upload_file(bucket_name,file_name, file_path)
    print(f"文件 {file_path} 已上传到桶 {bucket_name}")
else:
    print("未选择文件")

文件 C:/Users/george/Documents/乔治U盘/qzx3(2)/qzx(1)/(1)/PPT/青花瓷（2024.4.25）(小报).pptx 已上传到桶 share


In [5]:
# 验证环境变量配置
import os
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

print("=== Minio 环境变量配置验证 ===")
print(f"minio_ENDPOINT: {os.getenv('minio_ENDPOINT')}")
print(f"minio_ACCESS_KEY: {os.getenv('minio_ACCESS_KEY')}")
print(f"minio_SECRET_KEY: {os.getenv('minio_SECRET_KEY')}")
print(f"minio_CACHE_PATH: {os.getenv('minio_CACHE_PATH')}")

print("\n=== MinioClientWrapper 配置验证 ===")
print(f"客户端缓存路径: {minio_client.MINIO_CACHE_PATH}")
print(f"缓存目录是否存在: {os.path.exists(minio_client.MINIO_CACHE_PATH)}")

# 验证连接
try:
    buckets = minio_client.list_buckets()
    print(f"✓ 连接成功，发现 {len(buckets)} 个存储桶")
except Exception as e:
    print(f"✗ 连接失败: {e}")

=== Minio 环境变量配置验证 ===
minio_ENDPOINT: 192.168.1.201:19000
minio_ACCESS_KEY: A9LUdGKJSbSTyV9FrfsF
minio_SECRET_KEY: dyyJTGH2bhqXp2ziyIzbJ8os3GOuEbPDauvdMq60
minio_CACHE_PATH: c:\minio_cache

=== MinioClientWrapper 配置验证 ===
客户端缓存路径: c:\minio_cache
缓存目录是否存在: True
✓ 连接成功，发现 2 个存储桶


In [6]:
# 文件下载功能测试
import os

# 测试文件下载
bucket_name = "hdf5"
object_name = "akshare/data/1min/000001.SZ.h5"  # 选择一个存在的文件
local_file_name = "test_download.h5"
download_path = os.path.join(minio_client.MINIO_CACHE_PATH, local_file_name)

print(f"正在下载文件...")
print(f"桶名称: {bucket_name}")
print(f"对象名称: {object_name}")
print(f"本地路径: {download_path}")

# 检查文件是否存在
if minio_client.file_exists(bucket_name, object_name):
    print("✓ 文件存在于 Minio 中")
    
    # 下载文件
    success = minio_client.download_file(bucket_name, object_name, download_path)
    
    if success:
        print("✓ 文件下载成功")
        print(f"文件大小: {os.path.getsize(download_path)} 字节")
        
        # 清理测试文件
        if os.path.exists(download_path):
            os.remove(download_path)
            print("✓ 测试文件已清理")
    else:
        print("✗ 文件下载失败")
else:
    print("✗ 文件不存在于 Minio 中")

正在下载文件...
桶名称: hdf5
对象名称: akshare/data/1min/000001.SZ.h5
本地路径: c:\minio_cache\test_download.h5
✓ 文件存在于 Minio 中
✓ 文件下载成功
文件大小: 33173740 字节
✓ 测试文件已清理
